In [1]:
import sys
import os
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# List OpenAI models

In [15]:
from openai import OpenAI
client = OpenAI()
openAiModels = client.models.list()
for model in openAiModels:
    print(model.id)

gpt-3.5-turbo
gpt-5.4
gpt-5.3-chat-latest
gpt-5.4-2026-03-05
gpt-5.4-pro
gpt-5.4-pro-2026-03-05
davinci-002
babbage-002
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
dall-e-3
dall-e-2
gpt-3.5-turbo-1106
tts-1-hd
tts-1-1106
tts-1-hd-1106
text-embedding-3-small
text-embedding-3-large
gpt-3.5-turbo-0125
gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
gpt-4o-audio-preview
omni-moderation-latest
omni-moderation-2024-09-26
gpt-4o-audio-preview-2024-12-17
gpt-4o-mini-audio-preview-2024-12-17
o1-2024-12-17
o1
gpt-4o-mini-audio-preview
o3-mini
o3-mini-2025-01-31
gpt-4o-2024-11-20
gpt-4o-mini-search-preview-2025-03-11
gpt-4o-mini-search-preview
gpt-4o-transcribe
gpt-4o-mini-transcribe
gpt-4o-mini-tts
o3-2025-04-16
o4-mini-2025-04-16
o3
o4-mini
gpt-4.1-2025-04-14
gpt-4.1
gpt-4.1-mini-2025-04-14
gpt-4.1-mini
gpt-4.1-nano-2025-04-14
gpt-4.1-nano
gpt-image-1
gpt-4o-audio-preview-2025-06-03
gpt-4o-transcribe-diarize
gpt-5-chat-latest
gpt-5-2025-08-07
gpt-5
gpt-5-min

In [20]:
import aisuite as ai
client = ai.Client()
# MODEL="openai:gpt-4.1"
# MODEL="openai:gpt-4o"

In [21]:
# Setup Tavily Search Tool

In [35]:
from tavily import TavilyClient

def tavily_search_tool(query: str, max_results: int = 50, include_images: bool = False) -> list[dict]:
    """
    Perform a search using the Tavily API.

    Args:
        query (str): The search query.
        max_results (int): Number of results to return (default 5).
        include_images (bool): Whether to include image results.

    Returns:
        list[dict]: A list of dictionaries with keys like 'title', 'content', and 'url'.
    """
    params = {}
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        raise ValueError("TAVILY_API_KEY not found in environment variables.")
    params['api_key'] = api_key

    #client = TavilyClient(api_key)

    api_base_url = os.getenv("DLAI_TAVILY_BASE_URL")
    if api_base_url:
        params['api_base_url'] = api_base_url

    client = TavilyClient(api_key=api_key, api_base_url=api_base_url)

    try:
        response = client.search(
            query=query,
            max_results=max_results,
            include_images=include_images
        )

        results = []
        for r in response.get("results", []):
            results.append({
                "title": r.get("title", ""),
                "content": r.get("content", ""),
                "url": r.get("url", "")
            })

        if include_images:
            for img_url in response.get("images", []):
                results.append({"image_url": img_url})

        return results

    except Exception as e:
        return [{"error": str(e)}]  # For LLM-friendly agents
    

tavily_tool_def = {
    "type": "function",
    "function": {
        "name": "tavily_search_tool",
        "description": "Performs a general-purpose web search using the Tavily API.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search keywords for retrieving information from the web."
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results to return.",
                    "default": 50
                },
                "include_images": {
                    "type": "boolean",
                    "description": "Whether to include image results.",
                    "default": False
                }
            },
            "required": ["query"]
        }
    }
}






In [53]:
def collectProducts(vendor, prod_name, prod_url:str) -> str:
    prompt = f"""
    You are a market researcher.
    Go to the product webapge at: {prod_url} and retrieve the current listed price.

    check the amazon US price of the product "{prod_name}" form the vendor "{vendor}"
    
    You are a research function with access to:
    - tavily_tool: general web search (return JSON when asked)

    Return a strict JSON object with two fields:
    - "product_name": name of the product
    - "store_price": the price in the product page at the vendor's website
    - "amazon_price": the price found at the amazon store.
    - "amazon_url": the Amazon url for the product 
    """

    response = client.chat.completions.create(
        # model="openai:gpt-3.5-turbo-0125",
        model="openai:gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0,
        tools = [tavily_search_tool],
        tool_choice="auto",
        max_turns=5
    )

    
    content = response.choices[0].message.content
    print(content)
    return content

In [55]:
res = collectProducts("lelo", "liv 3", "https://www.lelo.com/liv-3")
res

```json
{
  "product_name": "Liv 3",
  "store_price": "Not explicitly found on the website, please check the product page for the most accurate pricing.",
  "amazon_price": "$139.00",
  "amazon_url": "https://www.amazon.com/LELO-App-Controlled-Vibrator-Waterproof-Vibrators/dp/B0D9YRPYFN"
}
``` 

Note: The price from the vendor's website wasn't explicitly found during the search.


'```json\n{\n  "product_name": "Liv 3",\n  "store_price": "Not explicitly found on the website, please check the product page for the most accurate pricing.",\n  "amazon_price": "$139.00",\n  "amazon_url": "https://www.amazon.com/LELO-App-Controlled-Vibrator-Waterproof-Vibrators/dp/B0D9YRPYFN"\n}\n``` \n\nNote: The price from the vendor\'s website wasn\'t explicitly found during the search.'